In [1]:
from google.colab import drive
drive.mount("/content/drive")

import os
import pandas as pd
import numpy as np
import re
from pathlib import Path

RUN_TAG = "random20k_nightlights_proxy_resnet18"

IMAGES_DIR = "/content/drive/MyDrive/india_2015_images/gee_chips_2015_random20k"
CLUSTERS_CSV = "/content/drive/MyDrive/IN2015_clusters_nightlights.csv"

OUT_DIR = "/content/drive/MyDrive/models_india_2015"
os.makedirs(OUT_DIR, exist_ok=True)

print("IMAGES_DIR exists:", os.path.isdir(IMAGES_DIR), IMAGES_DIR)
print("CLUSTERS_CSV exists:", os.path.exists(CLUSTERS_CSV), CLUSTERS_CSV)
print("OUT_DIR:", OUT_DIR)

Mounted at /content/drive
IMAGES_DIR exists: True /content/drive/MyDrive/india_2015_images/gee_chips_2015_random20k
CLUSTERS_CSV exists: True /content/drive/MyDrive/IN2015_clusters_nightlights.csv
OUT_DIR: /content/drive/MyDrive/models_india_2015


In [2]:
# Scan image files from filenames
pat = re.compile(
    r"^c(?P<cluster>\d+)_s(?P<sample>\d+)_lat(?P<lat>-?\d+(?:\.\d+)?)_lon(?P<lon>-?\d+(?:\.\d+)?)\.png$"
)

rows = []
for fn in os.listdir(IMAGES_DIR):
    m = pat.match(fn)
    if not m:
        continue
    rows.append({
        "image_name": fn,
        "cluster_id": int(m.group("cluster")),
        "sample_id": int(m.group("sample")),
        "lat": float(m.group("lat")),
        "lon": float(m.group("lon")),
        "img_path": os.path.join(IMAGES_DIR, fn),
    })

meta = pd.DataFrame(rows)
print("Found images:", len(meta))
print(meta.head())

Found images: 24583
                               image_name  cluster_id  sample_id       lat  \
0  c340405_s1_lat30.05521_lon80.16280.png      340405          1  30.05521   
1  c340405_s2_lat30.02609_lon80.22462.png      340405          2  30.02609   
2  c340405_s3_lat30.03423_lon80.21097.png      340405          3  30.03423   
3  c340405_s4_lat30.03942_lon80.15869.png      340405          4  30.03942   
4  c340405_s5_lat30.10125_lon80.17767.png      340405          5  30.10125   

        lon                                           img_path  
0  80.16280  /content/drive/MyDrive/india_2015_images/gee_c...  
1  80.22462  /content/drive/MyDrive/india_2015_images/gee_c...  
2  80.21097  /content/drive/MyDrive/india_2015_images/gee_c...  
3  80.15869  /content/drive/MyDrive/india_2015_images/gee_c...  
4  80.17767  /content/drive/MyDrive/india_2015_images/gee_c...  


## Merge nightlights labels and do group split



In [3]:
from sklearn.model_selection import GroupShuffleSplit

clusters = pd.read_csv(CLUSTERS_CSV)
clusters["cluster_id"] = clusters["cluster_id"].astype(int)

# Bin log1p nightlights into 5 quantiles
nl = clusters["nightlights_mean"].astype(float).to_numpy()
nl_log = np.log1p(nl)
ok = np.isfinite(nl_log)

clusters.loc[ok, "nightlights_bin"] = pd.qcut(
    nl_log[ok],
    q=5,
    labels=False,
    duplicates="drop"
).astype(int)

clusters["nightlights_bin"] = clusters["nightlights_bin"].astype("Int64")

# Merge image metadata with nightlights proxy labels
df = meta.merge(
    clusters[["cluster_id", "nightlights_mean", "nightlights_bin"]],
    on="cluster_id",
    how="left"
)

# Keep rows with valid labels
df = df.dropna(subset=["nightlights_bin"]).reset_index(drop=True)
df["label"] = df["nightlights_bin"].astype(int)

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, valid_idx = next(gss.split(df, groups=df["cluster_id"]))

train_df = df.iloc[train_idx].reset_index(drop=True)
valid_df = df.iloc[valid_idx].reset_index(drop=True)

N_CLASSES = int(df["label"].nunique())

## Dataloader



In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms

INPUT_SIZE = 224
BATCH_SIZE = 128

# Training data transforms
train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(INPUT_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

# Validation data transforms
valid_tfms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(INPUT_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

class ChipsDataset(Dataset):
    # Store dataframe and transforms
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    # Return dataset size
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["img_path"]).convert("RGB")
        x = self.transform(img)
        y = int(row["label"])
        return x, y

train_ds = ChipsDataset(train_df, train_tfms)
valid_ds = ChipsDataset(valid_df, valid_tfms)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Data loaders
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

valid_loader = DataLoader(
    valid_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

In [12]:
import torch.nn as nn
import torch.optim as optim
from torchvision import models

MODEL_NAME = "resnet18"

# ResNet18 backbone
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Replace classification head for proxy task
model.fc = nn.Linear(model.fc.in_features, N_CLASSES)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [13]:
from tqdm.auto import tqdm
import torch

EPOCHS = 8

best_acc = -1.0
best_path = os.path.join(OUT_DIR, f"cnn_{RUN_TAG}_best.pt")

def run_epoch(loader, train=True):
    model.train(train)
    total_loss, correct, total = 0.0, 0, 0

    for x, y in tqdm(loader, leave=False):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        if train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train):
            logits = model(x)
            loss = criterion(logits, y)
            if train:
                loss.backward()
                optimizer.step()

        total_loss += float(loss.item()) * x.size(0)
        pred = logits.argmax(dim=1)
        correct += int((pred == y).sum().item())
        total += x.size(0)

    return total_loss / total, correct / total

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(valid_loader, train=False)

    print(f"Epoch {epoch}/{EPOCHS} | train loss {tr_loss:.4f} acc {tr_acc:.3f} | val loss {va_loss:.4f} acc {va_acc:.3f}")

    if va_acc > best_acc:
        best_acc = va_acc
        torch.save({
            "run_tag": RUN_TAG,
            "model_name": MODEL_NAME,
            "state_dict": model.state_dict(),
            "input_size": INPUT_SIZE,
            "n_classes": N_CLASSES,
            "best_val_acc": best_acc,
            "label_type": "nightlights_bin (proxy task)",
        }, best_path)

print("Best val acc:", best_acc)
print("Saved:", best_path)

  0%|          | 0/153 [00:00<?, ?it/s]

  0%|          | 0/39 [00:00<?, ?it/s]

Epoch 1/8 | train loss 1.3383 acc 0.426 | val loss 1.3427 acc 0.418


  0%|          | 0/153 [00:00<?, ?it/s]

  0%|          | 0/39 [00:00<?, ?it/s]

...

  0%|          | 0/39 [00:00<?, ?it/s]

Epoch 7/8 | train loss 0.9822 acc 0.598 | val loss 1.4422 acc 0.457


  0%|          | 0/153 [00:00<?, ?it/s]

  0%|          | 0/39 [00:00<?, ?it/s]

Epoch 8/8 | train loss 0.9465 acc 0.616 | val loss 1.6087 acc 0.447
Best val acc: 0.45740365111561865
Saved: /content/drive/MyDrive/models_india_2015/cnn_random20k_nightlights_proxy_resnet18_best.pt
